In [1]:
print("how are you")


how are you


In [3]:
print("how are you")

how are you


In [4]:
from langchain_core.documents import Document
doc=Document(
    page_content="The actual text extracted from the PDF",
    metadata={
        "source_file": "Bharatiya_Nyaya_Sanhita_2023.pdf",
        "page_number": 10,
        "document_type": "central_act"
    }
)
print(doc)

page_content='The actual text extracted from the PDF' metadata={'source_file': 'Bharatiya_Nyaya_Sanhita_2023.pdf', 'page_number': 10, 'document_type': 'central_act'}


In [5]:
print(doc.page_content)

The actual text extracted from the PDF


In [6]:
print(doc.metadata)

{'source_file': 'Bharatiya_Nyaya_Sanhita_2023.pdf', 'page_number': 10, 'document_type': 'central_act'}


In [1]:
import pymupdf # PyMuPDF
import re
import unicodedata

from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

Current working directory:
/Users/sharondevaisa/Desktop/ccpeeps-legal-ai/services/rag/notebook


In [9]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

pdf_files = list(DATA_DIR.rglob("*.pdf"))

print("\nPDF files found:", len(pdf_files))

for pdf in pdf_files:
    print(pdf)

Project root: /Users/sharondevaisa/Desktop/ccpeeps-legal-ai/services/rag
Data directory: /Users/sharondevaisa/Desktop/ccpeeps-legal-ai/services/rag/data

PDF files found: 1
/Users/sharondevaisa/Desktop/ccpeeps-legal-ai/services/rag/data/Bharatiya_Nagarik_Suraksha_Sanhita_2023 (1).pdf


In [10]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

pdf_files = list(DATA_DIR.rglob("*.pdf"))

print("PDF files found:", len(pdf_files))

for pdf in pdf_files:
    print(pdf)

PDF files found: 1
/Users/sharondevaisa/Desktop/ccpeeps-legal-ai/services/rag/data/Bharatiya_Nagarik_Suraksha_Sanhita_2023 (1).pdf


In [11]:
PDF_PATH = pdf_files[0]

print("Selected PDF:", PDF_PATH)

Selected PDF: /Users/sharondevaisa/Desktop/ccpeeps-legal-ai/services/rag/data/Bharatiya_Nagarik_Suraksha_Sanhita_2023 (1).pdf


In [13]:
def extract_pdf_pages(pdf_path):

    pages = []

    with pymupdf.open(pdf_path) as pdf:

        for page_number, page in enumerate(pdf, start=1):

            text = page.get_text("text")

            pages.append({
                "page_number": page_number,
                "text": text
            })

    return pages

In [12]:
raw_pages = extract_pdf_pages(PDF_PATH)

print("Total pages:", len(raw_pages))

Total pages: 282


In [7]:
def clean_text(text):

    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Remove null characters
    text = text.replace("\x00", "")

    # Normalize line endings
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Replace tabs with spaces
    text = text.replace("\t", " ")

    # Remove excessive spaces
    text = re.sub(r"[ ]{2,}", " ", text)

    # Remove spaces at the beginning and end of lines
    text = "\n".join(
        line.strip()
        for line in text.split("\n")
    )

    # Reduce excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [14]:
cleaned_pages = []

for page in raw_pages:

    cleaned_text = clean_text(page["text"])

    cleaned_pages.append({
        "page_number": page["page_number"],
        "text": cleaned_text
    })

print("Cleaning completed.")

Cleaning completed.


In [15]:
for i in range(min(3, len(raw_pages))):
    print(f"\n--- PAGE {i + 1}: ORIGINAL ---")
    print(raw_pages[i]["text"][:1500])

    print(f"\n--- PAGE {i + 1}: CLEANED ---")
    print(cleaned_pages[i]["text"][:1500])


--- PAGE 1: ORIGINAL ---
1 
 
 
 
 
 
 
 
 
 
 
The Bharatiya Nagarik Suraksha Sanhita, 2023 
 (ACT NO. 46 OF 2023) 
[As on the 6th October, 2025] 
 
 
IndiaCode


--- PAGE 1: CLEANED ---
1

The Bharatiya Nagarik Suraksha Sanhita, 2023
(ACT NO. 46 OF 2023)
[As on the 6th October, 2025]

IndiaCode

--- PAGE 2: ORIGINAL ---
2 
 
LIST OF ABBREVIATIONS USED 
G.S.R.              . 
. 
. 
.            .            for           General Statutory Rules.  
S.O.                 . 
. 
. 
. 
.            ,,              Statutory Order. 
Notifn.  
. 
. 
. 
. 
.  
,, 
   Notification. 
 
 
IndiaCode


--- PAGE 2: CLEANED ---
2

LIST OF ABBREVIATIONS USED
G.S.R. .
.
.
. . for General Statutory Rules.
S.O. .
.
.
.
. ,, Statutory Order.
Notifn.
.
.
.
.
.
,,
Notification.

IndiaCode

--- PAGE 3: ORIGINAL ---
3 
 
THE BHARATIYA NAGARIK SURAKSHA SANHITA, 2023 
_____________ 
ARRANGEMENT OF SECTIONS 
__________ 
CHAPTER I 
PRELIMINARY 
SECTIONS 
1. Short title, extent and commencement. 
2. Definitions. 

In [16]:
for page in cleaned_pages[:5]:
    print(f"\n--- PAGE {page['page_number']} ---")
    print(page["text"][:2500])


--- PAGE 1 ---
1

The Bharatiya Nagarik Suraksha Sanhita, 2023
(ACT NO. 46 OF 2023)
[As on the 6th October, 2025]

IndiaCode

--- PAGE 2 ---
2

LIST OF ABBREVIATIONS USED
G.S.R. .
.
.
. . for General Statutory Rules.
S.O. .
.
.
.
. ,, Statutory Order.
Notifn.
.
.
.
.
.
,,
Notification.

IndiaCode

--- PAGE 3 ---
3

THE BHARATIYA NAGARIK SURAKSHA SANHITA, 2023
_____________
ARRANGEMENT OF SECTIONS
__________
CHAPTER I
PRELIMINARY
SECTIONS
1. Short title, extent and commencement.
2. Definitions.
3. Construction of references.
4. Trial of offences under Bharatiya Nyaya Sanhita, 2023 and other laws.
5. Saving.
CHAPTER II
CONSTITUTION OF CRIMINAL COURTS AND OFFICES

6. Classes of Criminal Courts.
7. Territorial divisions.
8. Court of Session.
9. Courts of Judicial Magistrates.
10. Chief Judicial Magistrate and Additional Chief Judicial Magistrate, etc.
11. Special Judicial Magistrates.
12. Local Jurisdiction of Judicial Magistrates.
13. Subordination of Judicial Magistrates.
14. Executive 

In [17]:
for page in cleaned_pages:
    text = page["text"]

    if "CHAPTER I" in text or "1." in text:
        print(f"\n--- PAGE {page['page_number']} ---")
        print(text[:3000])


--- PAGE 3 ---
3

THE BHARATIYA NAGARIK SURAKSHA SANHITA, 2023
_____________
ARRANGEMENT OF SECTIONS
__________
CHAPTER I
PRELIMINARY
SECTIONS
1. Short title, extent and commencement.
2. Definitions.
3. Construction of references.
4. Trial of offences under Bharatiya Nyaya Sanhita, 2023 and other laws.
5. Saving.
CHAPTER II
CONSTITUTION OF CRIMINAL COURTS AND OFFICES

6. Classes of Criminal Courts.
7. Territorial divisions.
8. Court of Session.
9. Courts of Judicial Magistrates.
10. Chief Judicial Magistrate and Additional Chief Judicial Magistrate, etc.
11. Special Judicial Magistrates.
12. Local Jurisdiction of Judicial Magistrates.
13. Subordination of Judicial Magistrates.
14. Executive Magistrates.
15. Special Executive Magistrates.
16. Local Jurisdiction of Executive.
17. Subordination of Executive Magistrates.
18. Public Prosecutors.
19. Assistant Public Prosecutors.
20. Directorate of Prosecution.
CHAPTER III
POWER OF COURTS
21. Courts by which offences are triable.
22. Senten

In [18]:
for page in cleaned_pages:
    text = page["text"]

    if "Short title, extent and commencement" in text:
        print(f"\n--- PAGE {page['page_number']} ---")
        print(text[:3000])


--- PAGE 3 ---
3

THE BHARATIYA NAGARIK SURAKSHA SANHITA, 2023
_____________
ARRANGEMENT OF SECTIONS
__________
CHAPTER I
PRELIMINARY
SECTIONS
1. Short title, extent and commencement.
2. Definitions.
3. Construction of references.
4. Trial of offences under Bharatiya Nyaya Sanhita, 2023 and other laws.
5. Saving.
CHAPTER II
CONSTITUTION OF CRIMINAL COURTS AND OFFICES

6. Classes of Criminal Courts.
7. Territorial divisions.
8. Court of Session.
9. Courts of Judicial Magistrates.
10. Chief Judicial Magistrate and Additional Chief Judicial Magistrate, etc.
11. Special Judicial Magistrates.
12. Local Jurisdiction of Judicial Magistrates.
13. Subordination of Judicial Magistrates.
14. Executive Magistrates.
15. Special Executive Magistrates.
16. Local Jurisdiction of Executive.
17. Subordination of Executive Magistrates.
18. Public Prosecutors.
19. Assistant Public Prosecutors.
20. Directorate of Prosecution.
CHAPTER III
POWER OF COURTS
21. Courts by which offences are triable.
22. Senten

In [19]:

import re

LEGAL_UNIT_PATTERN = re.compile(
    r"(?im)^\s*"
    r"(?P<label>"
    r"SECTION|SECTIONS|ARTICLE|ARTICLES|RULE|RULES|"
    r"REGULATION|REGULATIONS|CLAUSE|ORDER|"
    r"CHAPTER|PART|"
    r"\d+[A-Z]?(?:\([A-Za-z0-9]+\))*\.?"
    r")"
    r"\s+"
    r"(?P<number>[A-Z0-9IVXLCDM]+(?:\([A-Za-z0-9]+\))*)?"
    r"\s*"
    r"(?P<title>[^\n]*)"
)

In [20]:
print("Variables available:")
print([name for name in globals() if name in ["pages", "full_text", "cleaned_pages", "text"]])

if "pages" in globals():
    print("Pages:", len(pages))
    print("\nFirst page sample:\n", str(pages[0])[:1000])
elif "cleaned_pages" in globals():
    print("Pages:", len(cleaned_pages))
    print("\nFirst page sample:\n", str(cleaned_pages[0])[:1000])
elif "full_text" in globals():
    print("\nFull text sample:\n", full_text[:1000])
elif "text" in globals():
    print("\nText sample:\n", text[:1000])

Variables available:
['cleaned_pages', 'text']
Pages: 282

First page sample:
 {'page_number': 1, 'text': '1\n\nThe Bharatiya Nagarik Suraksha Sanhita, 2023\n(ACT NO. 46 OF 2023)\n[As on the 6th October, 2025]\n\nIndiaCode'}


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [21]:
for page in cleaned_pages:
    page_text = page["text"]

    if "BE it enacted by Parliament" in page_text:
        print("Found enactment text on PDF page:", page["page_number"])
        print(page_text[:2000])
        break

Found enactment text on PDF page: 18
18

THE BHARATIYA NAGARIK SURAKSHA SANHITA, 2023
ACT NO. 46 OF 2023
[25th December, 2023.]
An Act to consolidate and amend the law relating to Criminal Procedure.
BE it enacted by Parliament in the Seventy-fourth Year of the Republic of India as follows:—
CHAPTER I
PRELIMINARY
1. Short title, extent and commencement.—(1) This Act may be called the Bharatiya Nagarik
Suraksha Sanhita, 2023.
(2) The provisions of this Sanhita, other than those relating to Chapters IX, XI and XII thereof, shall
not apply—
(a) to the State of Nagaland;
(b) to the tribal areas,
but the concerned State Government may, by notification, apply such provisions or any of them
to the whole or part of the State of Nagaland or such tribal areas, as the case may be, with such
supplemental, incidental or consequential modifications, as may be specified in the notification.
Explanation.—In this section, “tribal areas” means the territories which immediately before
the 21st day of Jan

In [22]:
import re

for page in cleaned_pages:
    page_text = page["text"]
    lines = [line.strip() for line in page_text.splitlines() if line.strip()]

    # Count numbered heading-like lines
    numbered_headings = [
        line for line in lines
        if re.match(r"^\d+[A-Z]?(?:\([a-zA-Z0-9]+\))?\.?\s+[A-Z]", line)
    ]

    # Detect pages with structural headings
    structural_headings = [
        line for line in lines
        if re.match(
            r"^(CHAPTER|PART|SECTION|ARTICLE|SCHEDULE)\b",
            line,
            re.IGNORECASE
        )
    ]

    if len(numbered_headings) >= 3 or structural_headings:
        print(f"\n--- PDF page {page['page_number']} ---")
        print("Numbered headings:", numbered_headings[:5])
        print("Structural headings:", structural_headings[:5])


--- PDF page 3 ---
Numbered headings: ['1. Short title, extent and commencement.', '2. Definitions.', '3. Construction of references.', '4. Trial of offences under Bharatiya Nyaya Sanhita, 2023 and other laws.', '5. Saving.']
Structural headings: ['CHAPTER I', 'CHAPTER II', 'CHAPTER III']

--- PDF page 4 ---
Numbered headings: ['30. Powers of superior officers of police.', '31. Public when to assist Magistrates and police.', '32. Aid to person, other than police officer, executing warrant.', '33. Public to give information of certain offences.', '34. Duty of officers employed in connection with affairs of a village to make certain']
Structural headings: ['CHAPTER IV', 'CHAPTER V', 'CHAPTER VI']

--- PDF page 5 ---
Numbered headings: ['69. Service of summons outside local limits.', '70. Proof of service in such cases and when serving officer not present.', '71. Service of summons on witness.', '72. Form of warrant of arrest and duration.', '73. Power to direct security to be taken.']
S

In [23]:
import re

# Common legal heading patterns
TOP_LEVEL_PATTERNS = [
    re.compile(r"^\s*(CHAPTER)\s+([IVXLCDM]+|\d+)\b\s*(.*)$", re.IGNORECASE),
    re.compile(r"^\s*(PART)\s+([IVXLCDM]+|\d+)\b\s*(.*)$", re.IGNORECASE),
    re.compile(r"^\s*(TITLE)\s+([IVXLCDM]+|\d+)\b\s*(.*)$", re.IGNORECASE),
    re.compile(r"^\s*(ARTICLE)\s+(\d+[A-Z]?)\b\s*(.*)$", re.IGNORECASE),
]

PROVISION_PATTERN = re.compile(
    r"^\s*(\d+[A-Z]?)\.\s+([A-Z][^\n]{0,180})$"
)

def detect_headings(page_text):
    """
    Detect possible top-level legal headings and numbered provisions
    from the text of a single page.
    """
    detected = []

    for line in page_text.splitlines():
        line = line.strip()

        if not line:
            continue

        # Check top-level headings first
        for pattern in TOP_LEVEL_PATTERNS:
            match = pattern.match(line)

            if match:
                detected.append({
                    "type": match.group(1).upper(),
                    "number": match.group(2),
                    "title": match.group(3).strip(),
                    "line": line
                })
                break
        else:
            # If it is not a top-level heading, check numbered provisions
            match = PROVISION_PATTERN.match(line)

            if match:
                detected.append({
                    "type": "PROVISION",
                    "number": match.group(1),
                    "title": match.group(2).strip(),
                    "line": line
                })

    return detected

In [24]:
for page in cleaned_pages:
    headings = detect_headings(page["text"])

    if headings:
        print(f"\n--- PDF page {page['page_number']} ---")

        for heading in headings:
            print(heading)


--- PDF page 3 ---
{'type': 'CHAPTER', 'number': 'I', 'title': '', 'line': 'CHAPTER I'}
{'type': 'PROVISION', 'number': '1', 'title': 'Short title, extent and commencement.', 'line': '1. Short title, extent and commencement.'}
{'type': 'PROVISION', 'number': '2', 'title': 'Definitions.', 'line': '2. Definitions.'}
{'type': 'PROVISION', 'number': '3', 'title': 'Construction of references.', 'line': '3. Construction of references.'}
{'type': 'PROVISION', 'number': '4', 'title': 'Trial of offences under Bharatiya Nyaya Sanhita, 2023 and other laws.', 'line': '4. Trial of offences under Bharatiya Nyaya Sanhita, 2023 and other laws.'}
{'type': 'PROVISION', 'number': '5', 'title': 'Saving.', 'line': '5. Saving.'}
{'type': 'CHAPTER', 'number': 'II', 'title': '', 'line': 'CHAPTER II'}
{'type': 'PROVISION', 'number': '6', 'title': 'Classes of Criminal Courts.', 'line': '6. Classes of Criminal Courts.'}
{'type': 'PROVISION', 'number': '7', 'title': 'Territorial divisions.', 'line': '7. Territor

In [25]:
for page in cleaned_pages[:20]:
    print(f"\n--- PDF page {page['page_number']} ---")
    print(page["text"][:1200])


--- PDF page 1 ---
1

The Bharatiya Nagarik Suraksha Sanhita, 2023
(ACT NO. 46 OF 2023)
[As on the 6th October, 2025]

IndiaCode

--- PDF page 2 ---
2

LIST OF ABBREVIATIONS USED
G.S.R. .
.
.
. . for General Statutory Rules.
S.O. .
.
.
.
. ,, Statutory Order.
Notifn.
.
.
.
.
.
,,
Notification.

IndiaCode

--- PDF page 3 ---
3

THE BHARATIYA NAGARIK SURAKSHA SANHITA, 2023
_____________
ARRANGEMENT OF SECTIONS
__________
CHAPTER I
PRELIMINARY
SECTIONS
1. Short title, extent and commencement.
2. Definitions.
3. Construction of references.
4. Trial of offences under Bharatiya Nyaya Sanhita, 2023 and other laws.
5. Saving.
CHAPTER II
CONSTITUTION OF CRIMINAL COURTS AND OFFICES

6. Classes of Criminal Courts.
7. Territorial divisions.
8. Court of Session.
9. Courts of Judicial Magistrates.
10. Chief Judicial Magistrate and Additional Chief Judicial Magistrate, etc.
11. Special Judicial Magistrates.
12. Local Jurisdiction of Judicial Magistrates.
13. Subordination of Judicial Magistrates.
14

In [26]:
for page in cleaned_pages[:15]:
    text = page["text"]

    print(
        f"PDF page {page['page_number']} | "
        f"{len(text)} characters | "
        f"{text[:180].replace(chr(10), ' ')}"
    )

PDF page 1 | 109 characters | 1  The Bharatiya Nagarik Suraksha Sanhita, 2023 (ACT NO. 46 OF 2023) [As on the 6th October, 2025]  IndiaCode
PDF page 2 | 156 characters | 2  LIST OF ABBREVIATIONS USED G.S.R. . . . . . for General Statutory Rules. S.O. . . . . . ,, Statutory Order. Notifn. . . . . . ,, Notification.  IndiaCode
PDF page 3 | 1378 characters | 3  THE BHARATIYA NAGARIK SURAKSHA SANHITA, 2023 _____________ ARRANGEMENT OF SECTIONS __________ CHAPTER I PRELIMINARY SECTIONS 1. Short title, extent and commencement. 2. Definiti
PDF page 4 | 2266 characters | 4  CHAPTER IV POWERS OF SUPERIOR OFFICERS OF POLICE AND AID TO THE MAGISTRATES AND THE POLICE SECTIONS 30. Powers of superior officers of police. 31. Public when to assist Magistra
PDF page 5 | 2019 characters | 5  SECTIONS 69. Service of summons outside local limits. 70. Proof of service in such cases and when serving officer not present. 71. Service of summons on witness. B.—Warrant of a
PDF page 6 | 2127 characters | 6  C.—

In [27]:
for page in cleaned_pages:
    if page["page_number"] == 17:
        print(page["text"][:2500])
        break
    

17

SECTIONS
499. Payment to innocent purchaser of money found on accused.
500. Appeal against orders under section 498 or section 499.
501. Destruction of libellous and other matter.
502. Power to restore possession of immovable property.
503. Procedure by police upon seizure of property.
504. Procedure where no claimant appears within six months.
505. Power to sell perishable property.
CHAPTER XXXVII
IRREGULAR PROCEEDINGS
506. Irregularities which do not vitiate proceedings.
507. Irregularities which vitiate proceedings.
508. Proceedings in wrong place.
509. Non-compliance with provisions of section 183 or section 316.
510. Effect of omission to frame, or absence of, or error in, charge.
511. Finding or sentence when reversible by reason of error, omission or irregularity.
512. Defect or error not to make attachment unlawful.
CHAPTER XXXVIII
LIMITATION FOR TAKING COGNIZANCE OF CERTAIN OFFENCES
513. Definitions.
514. Bar to taking cognizance after lapse of period of limitation.
515. C

In [28]:
for page in cleaned_pages[15:25]:
    print(f"\n--- PDF page {page['page_number']} ---")
    print(page["text"][:700])


--- PDF page 16 ---
16

C.—Levy of fine
SECTIONS
461. Warrant for levy of fine.
462. Effect of such warrant.
463. Warrant for levy of fine issued by a Court in any territory to which this Sanhita does not extend.
464. Suspension of execution of sentence of imprisonment.
D.—General provisions regarding execution
465. Who may issue warrant.
466. Sentence on escaped convict when to take effect.
467. Sentence on offender already sentenced for another offence.
468. Period of detention undergone by accused to be set off against sentence of imprisonment.
469. Saving.
470. Return of warrant on execution of sentence.
471. Money ordered to be paid recoverable as a fine.
E.—Suspension, remission and commutation of sentenc

--- PDF page 17 ---
17

SECTIONS
499. Payment to innocent purchaser of money found on accused.
500. Appeal against orders under section 498 or section 499.
501. Destruction of libellous and other matter.
502. Power to restore possession of immovable property.
503. Procedure by

Gemini client initialized successfully


In [ ]:
from pathlib import Path

PDF_PATH = Path("../data/Bharatiya_Nagarik_Suraksha_Sanhita_2023 (1).pdf")

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH.resolve()}")

print("PDF found:", PDF_PATH.resolve())

PDF found: /Users/sharondevaisa/Desktop/ccpeeps-legal-ai/services/rag/data/Bharatiya_Nagarik_Suraksha_Sanhita_2023 (1).pdf


In [34]:
from dotenv import load_dotenv, find_dotenv
from google import genai
import os

print("STEP 1: Loading environment variables...")

env_path = find_dotenv()

if not env_path:
    raise FileNotFoundError(".env file not found")

load_dotenv(env_path)

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY is missing from .env")

print("API key loaded successfully.")

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully.")

STEP 1: Loading environment variables...
API key loaded successfully.
Gemini client initialized successfully.


In [35]:
from pathlib import Path

print("STEP 2: Checking the input PDF...")

PDF_PATH = Path(
    "/Users/sharondevaisa/Desktop/ccpeeps-legal-ai/services/rag/data/"
    "Bharatiya_Nagarik_Suraksha_Sanhita_2023 (1).pdf"
)

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")

print("PDF found:", PDF_PATH.name)
print("File size:", round(PDF_PATH.stat().st_size / (1024 * 1024), 2), "MB")

STEP 2: Checking the input PDF...
PDF found: Bharatiya_Nagarik_Suraksha_Sanhita_2023 (1).pdf
File size: 2.74 MB


In [41]:
from google import genai
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:

    raise ValueError("GEMINI_API_KEY not found in .env file")

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


In [39]:
from typing import List
import json
from pydantic import BaseModel
class DocumentMetadata(BaseModel):
    title: str
    document_type: str
    jurisdiction: str
    document_description: str


class ContentBoundaries(BaseModel):
    content_start_page: int
    content_end_page: int


class HierarchyLevel(BaseModel):
    level: int
    name: str
    marker_type: str


class DocumentStructureProfile(BaseModel):
    document_metadata: DocumentMetadata
    content_boundaries: ContentBoundaries
    hierarchy: List[HierarchyLevel]

In [42]:
import json

print(
    json.dumps(
        DocumentStructureProfile.model_json_schema(),
        indent=2
    )
)


{
  "$defs": {
    "ContentBoundaries": {
      "properties": {
        "content_start_page": {
          "title": "Content Start Page",
          "type": "integer"
        },
        "content_end_page": {
          "title": "Content End Page",
          "type": "integer"
        }
      },
      "required": [
        "content_start_page",
        "content_end_page"
      ],
      "title": "ContentBoundaries",
      "type": "object"
    },
    "DocumentMetadata": {
      "properties": {
        "title": {
          "title": "Title",
          "type": "string"
        },
        "document_type": {
          "title": "Document Type",
          "type": "string"
        },
        "jurisdiction": {
          "title": "Jurisdiction",
          "type": "string"
        },
        "document_description": {
          "title": "Document Description",
          "type": "string"
        }
      },
      "required": [
        "title",
        "document_type",
        "jurisdiction",
        "docum

In [43]:
prompt = """
You are a legal document structure analysis system.

Analyze the entire uploaded legal PDF and generate a structured JSON
profile describing its actual structure.

The output must match the provided Pydantic schema.

==================================================
1. DOCUMENT METADATA
==================================================

Identify the following:

- title: The official title of the document.
- document_type: The type of legal document, such as Central Act,
  Constitution, Amendment Act, Rules, Regulations, or another appropriate type.
- jurisdiction: The jurisdiction applicable to the document.
- document_description: A concise description of the document's
  purpose, scope, and subject matter.

The document_description will later be provided to an LLM as context
when answering questions using retrieved legal chunks.

==================================================
2. CONTENT BOUNDARIES
==================================================

Identify:

- content_start_page: The page where the actual legal content begins.
- content_end_page: The page where the actual legal content ends.

Do not mistake the table of contents, preface, or index for the
actual legal content.

Use the PDF's physical page numbers, starting from page 1.

==================================================
3. HIERARCHY IDENTIFICATION
==================================================

Identify the actual structural hierarchy of the legal document.

Possible hierarchy levels include:

- Part
- Chapter
- Section
- Subsection
- Clause
- Subclause
- Schedule

These are examples only. Do not assume that every document contains
all these levels.

RULES:

- Identify only hierarchy levels that genuinely exist.
- The hierarchy must be a dynamic JSON array.
- Do not assume a fixed number of hierarchy levels.
- If only one hierarchy level exists, return one hierarchy item.
- If three hierarchy levels exist, return three hierarchy items.
- Start numbering from level 1.
- Use sequential level numbers.
- Arrange hierarchy levels from highest to lowest.
- Do not invent hypothetical or missing levels.
- Identify the lowest actual hierarchy level.
- Analyze the actual document content, not only its table of contents.
- Avoid treating ordinary text, examples, or references as hierarchy levels.
- Distinguish genuine structural markers from similar markers appearing
  inside the body text.

For each hierarchy level, return:

- level: Sequential integer indicating its position in the hierarchy.
- name: Name of the structural unit.
- marker_type: The format used to identify that unit in the document.

Allowed marker_type values:

- named_heading
- numbered_heading
- parenthesized_number
- parenthesized_alphabet
- parenthesized_roman
- numbered_roman
- other

Examples:

"CHAPTER II" -> numbered_heading
"Section 5" -> numbered_heading
"(1)" -> parenthesized_number
"(a)" -> parenthesized_alphabet
"(i)" -> parenthesized_roman

These examples are illustrative. Use the actual document structure.

Do not return regex patterns.
Do not return marker descriptions.
Do not include extra fields.

==================================================
4. EXAMPLE OF EXPECTED JSON OUTPUT
==================================================

The following is an example of the expected JSON format:

{
  "document_metadata": {
    "title": "Example Legal Act",
    "document_type": "Central Act",
    "jurisdiction": "India",
    "document_description": "An Act establishing legal provisions related to a particular subject."
  },
  "content_boundaries": {
    "content_start_page": 5,
    "content_end_page": 120
  },
  "hierarchy": [
    {
      "level": 1,
      "name": "Chapter",
      "marker_type": "numbered_heading"
    },
    {
      "level": 2,
      "name": "Section",
      "marker_type": "numbered_heading"
    },
    {
      "level": 3,
      "name": "Subsection",
      "marker_type": "parenthesized_number"
    }
  ]
}

IMPORTANT ABOUT THE EXAMPLE:

- This example is only to demonstrate the JSON format.
- Do not copy its values.
- Analyze the actual uploaded PDF.
- The number of hierarchy items must depend on the actual document.
- A document with one hierarchy level must return only one item.
- A document with two hierarchy levels must return only two items.
- Do not assume that every document has three levels.
- The lowest hierarchy level must be the deepest genuine structural
  level identified in the document.

==================================================
5. OUTPUT REQUIREMENTS
==================================================

Return only valid JSON matching the Pydantic schema.

Do not include Markdown code fences.
Do not include explanations outside the JSON.
Do not include additional fields.
Do not return an empty object.
Use the actual content of the uploaded PDF to populate every field.

Ensure that all required fields are present and have the correct types.
"""

In [ ]:
uploaded_pdf = client.files.upload(
    file=str(PDF_PATH)
)

print("PDF uploaded successfully.")
print("Gemini file name:", uploaded_pdf.name)
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        prompt,
        uploaded_pdf
    ],
    config={
        "response_mime_type": "application/json",
        "response_schema": DocumentStructureProfile
    }
)

print("Gemini analysis completed.")

In [57]:
response_text = """
{
  "document_metadata": {
    "title": "The Bharatiya Nagarik Suraksha Sanhita, 2023",
    "document_type": "Central Act",
    "jurisdiction": "India",
    "document_description": "The Bharatiya Nagarik Suraksha Sanhita, 2023 (Act No. 46 of 2023) consolidates and amends the law relating to criminal procedure in India."
  },
  "content_boundaries": {
    "content_start_page": 18,
    "content_end_page": 274
  },
  "hierarchy": [
    {
      "level": 1,
      "name": "Chapter",
      "marker_type": "named_heading"
    },
    {
      "level": 2,
      "name": "Section",
      "marker_type": "numbered_heading"
    },
    {
      "level": 3,
      "name": "Sub-section",
      "marker_type": "parenthesized_number"
    },
    {
      "level": 4,
      "name": "Clause",
      "marker_type": "parenthesized_alphabet"
    },
    {
      "level": 5,
      "name": "Sub-clause",
      "marker_type": "parenthesized_roman"
    }
  ]
}
"""

structure_profile = DocumentStructureProfile.model_validate_json(response_text)

print(structure_profile.model_dump_json(indent=2))

{
  "document_metadata": {
    "title": "The Bharatiya Nagarik Suraksha Sanhita, 2023",
    "document_type": "Central Act",
    "jurisdiction": "India",
    "document_description": "The Bharatiya Nagarik Suraksha Sanhita, 2023 (Act No. 46 of 2023) consolidates and amends the law relating to criminal procedure in India."
  },
  "content_boundaries": {
    "content_start_page": 18,
    "content_end_page": 274
  },
  "hierarchy": [
    {
      "level": 1,
      "name": "Chapter",
      "marker_type": "named_heading"
    },
    {
      "level": 2,
      "name": "Section",
      "marker_type": "numbered_heading"
    },
    {
      "level": 3,
      "name": "Sub-section",
      "marker_type": "parenthesized_number"
    },
    {
      "level": 4,
      "name": "Clause",
      "marker_type": "parenthesized_alphabet"
    },
    {
      "level": 5,
      "name": "Sub-clause",
      "marker_type": "parenthesized_roman"
    }
  ]
}


In [45]:
from IPython.display import JSON, display

display(JSON(structure_profile.model_dump()))

<IPython.core.display.JSON object>

In [46]:
for item in structure_profile.hierarchy:
    print(
        f"Level {item.level}: {item.name} "
        f"({item.marker_type})"
    )

Level 1: Chapter (named_heading)
Level 2: Section (numbered_heading)
Level 3: Sub-section (parenthesized_number)
Level 4: Clause (parenthesized_alphabet)
Level 5: Sub-clause (parenthesized_roman)
Level 6: Further nested provision (other)


In [47]:
import pymupdf

PDF_PATH = "/Users/sharondevaisa/Desktop/ccpeeps-legal-ai/services/rag/data/Bharatiya_Nagarik_Suraksha_Sanhita_2023 (1).pdf"

doc = pymupdf.open(PDF_PATH)

start_page = structure_profile.content_boundaries.content_start_page
end_page = structure_profile.content_boundaries.content_end_page

pages = []

for page_num in range(start_page - 1, end_page):
    page = doc[page_num]

    pages.append({
        "page_number": page_num + 1,
        "text": page.get_text("text")
    })

print(f"Extracted {len(pages)} pages")

Extracted 257 pages


In [48]:
print(pages[0]["text"][:2000])

18 
 
THE BHARATIYA NAGARIK SURAKSHA SANHITA, 2023 
ACT NO. 46 OF 2023 
[25th December, 2023.] 
An Act to consolidate and amend the law relating to Criminal Procedure. 
BE it enacted by Parliament in the Seventy-fourth Year of the Republic of India as follows:— 
CHAPTER I 
PRELIMINARY 
1. Short title, extent and commencement.—(1) This Act may be called the Bharatiya Nagarik 
Suraksha Sanhita, 2023. 
(2) The provisions of this Sanhita, other than those relating to Chapters IX, XI and XII thereof, shall 
not apply— 
(a) to the State of Nagaland; 
(b) to the tribal areas, 
but the concerned State Government may, by notification, apply such provisions or any of them 
to the whole or part of the State of Nagaland or such tribal areas, as the case may be, with such 
supplemental, incidental or consequential modifications, as may be specified in the notification. 
Explanation.—In this section, “tribal areas” means the territories which immediately before                           
the 21st da

In [49]:
import tiktoken

# Tokenizer for counting tokens
tokenizer = tiktoken.get_encoding("cl100k_base")

# Maximum tokens allowed in a chunk
MAX_TOKENS = 800

# Number of overlapping tokens for the final fallback split
CHUNK_OVERLAP = 100


def count_tokens(text):
    return len(tokenizer.encode(text))


print("Maximum tokens:", MAX_TOKENS)
print("Overlap:", CHUNK_OVERLAP)

Maximum tokens: 800
Overlap: 100


In [55]:
import re


def split_pages_by_level(pages, level_name):

    # Find hierarchy information
    level_info = next(
        (
            level for level in structure_profile.hierarchy
            if level.name.lower() == level_name.lower()
        ),
        None
    )

    if level_info is None:
        raise ValueError(f"Level '{level_name}' not found")

    marker_type = level_info.marker_type

    # Create regex based on marker type
    if marker_type == "named_heading":
        pattern = re.compile(
            rf"(?m)^\s*{re.escape(level_name)}\s+[IVXLCDM]+\b[^\n]*"
        )

    elif marker_type == "numbered_heading":
        pattern = re.compile(
            r"(?m)^\s*\d{1,3}\.\s+[^\n]+"
        )

    elif marker_type == "parenthesized_number":
        pattern = re.compile(
            r"(?m)^\s*\(\d+\)\s+"
        )

    elif marker_type == "parenthesized_alphabet":
        pattern = re.compile(
            r"(?m)^\s*\([a-z]\)\s+"
        )

    elif marker_type == "parenthesized_roman":
        pattern = re.compile(
            r"(?m)^\s*\([ivxlcdm]+\)\s+"
        )

    else:
        raise ValueError(f"Unsupported marker type: {marker_type}")

    split_results = []

    current_chunk = ""
    current_start_page = None
    current_end_page = None
    current_pages = []

    def save_current_chunk():

        if current_chunk.strip():

            split_results.append({
                "start_page": current_start_page,
                "end_page": current_end_page,
                "level": level_name,
                "text": current_chunk.strip(),
                "pages": current_pages.copy()
            })

    for page in pages:

        text = page["text"]
        page_number = page["page_number"]

        matches = list(pattern.finditer(text))

        # No matching heading on this page.
        # Continue accumulating into the current chunk.
        if not matches:

            if text.strip():

                if current_start_page is None:
                    current_start_page = page_number

                current_chunk += "\n" + text.strip()
                current_end_page = page_number

                current_pages.append({
                    "page_number": page_number,
                    "text": text.strip()
                })

            continue

        # Add text before the first matching heading
        # to the previously accumulated chunk.
        text_before = text[:matches[0].start()].strip()

        if text_before:

            if current_start_page is None:
                current_start_page = page_number

            current_chunk += "\n" + text_before
            current_end_page = page_number

            current_pages.append({
                "page_number": page_number,
                "text": text_before
            })

        # Save only when a same-level heading is encountered.
        save_current_chunk()

        # Reset for the next hierarchy unit.
        current_chunk = ""
        current_start_page = None
        current_end_page = None
        current_pages = []

        # Process all matching headings on this page.
        for i, match in enumerate(matches):

            start = match.start()

            end = (
                matches[i + 1].start()
                if i + 1 < len(matches)
                else len(text)
            )

            chunk_text = text[start:end].strip()

            if not chunk_text:
                continue

            # Save previous same-level unit if one exists.
            save_current_chunk()

            # Start a new hierarchy unit.
            current_chunk = chunk_text
            current_start_page = page_number
            current_end_page = page_number

            current_pages = [{
                "page_number": page_number,
                "text": chunk_text
            }]

    # Save the final accumulated chunk at document end.
    save_current_chunk()

    return split_results

In [52]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

MAX_TOKENS = 800
CHUNK_OVERLAP = 100

def count_tokens(text):
    return len(tokenizer.encode(text))

In [50]:
def split_by_tokens(text, max_tokens=MAX_TOKENS, overlap=CHUNK_OVERLAP):

    tokens = tokenizer.encode(text)
    chunks = []

    start = 0

    while start < len(tokens):

        end = min(start + max_tokens, len(tokens))

        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens)

        chunks.append(chunk_text)

        if end == len(tokens):
            break

        start = end - overlap

    return chunks

In [53]:
import uuid


def recursive_chunking(
    pages,
    hierarchy,
    document_id,
    parent_id=None,
    level_index=0
):
    chunks = []

    level_name = hierarchy[level_index].name
    is_last_level = level_index == len(hierarchy) - 1

    # Split pages using the current hierarchy level
    level_chunks = split_pages_by_level(pages, level_name)

    for level_chunk in level_chunks:

        text = level_chunk["text"]
        token_count = count_tokens(text)

        # -----------------------------------------
        # LAST HIERARCHY LEVEL
        # -----------------------------------------
        if is_last_level:

            # If within the limit, keep the chunk unchanged
            if token_count <= MAX_TOKENS:

                chunks.append({
                    "document_id": document_id,
                    "chunk_id": str(uuid.uuid4()),
                    "parent_id": parent_id,
                    "level": level_name,
                    "start_page": level_chunk["start_page"],
                    "end_page": level_chunk["end_page"],
                    "text": text,
                    "token_count": token_count
                })

            # If oversized, split only this chunk by tokens
            else:

                parent_chunk_id = str(uuid.uuid4())

                chunks.append({
                    "document_id": document_id,
                    "chunk_id": parent_chunk_id,
                    "parent_id": parent_id,
                    "level": level_name,
                    "start_page": level_chunk["start_page"],
                    "end_page": level_chunk["end_page"],
                    "text": text,
                    "token_count": token_count
                })

                token_chunks = split_by_tokens(
                    text,
                    max_tokens=MAX_TOKENS,
                    overlap=CHUNK_OVERLAP
                )

                for token_text in token_chunks:

                    chunks.append({
                        "document_id": document_id,
                        "chunk_id": str(uuid.uuid4()),
                        "parent_id": parent_chunk_id,
                        "level": "token",
                        "start_page": level_chunk["start_page"],
                        "end_page": level_chunk["end_page"],
                        "text": token_text,
                        "token_count": count_tokens(token_text)
                    })

        # -----------------------------------------
        # INTERMEDIATE HIERARCHY LEVEL
        # -----------------------------------------
        else:

            # If oversized, create a parent and recurse
            if token_count > MAX_TOKENS:

                parent_chunk_id = str(uuid.uuid4())

                chunks.append({
                    "document_id": document_id,
                    "chunk_id": parent_chunk_id,
                    "parent_id": parent_id,
                    "level": level_name,
                    "start_page": level_chunk["start_page"],
                    "end_page": level_chunk["end_page"],
                    "text": text,
                    "token_count": token_count
                })

                # Recursively call the NEXT hierarchy level
                child_chunks = recursive_chunking(
                    pages=level_chunk["pages"],
                    hierarchy=hierarchy,
                    document_id=document_id,
                    parent_id=parent_chunk_id,
                    level_index=level_index + 1
                )

                chunks.extend(child_chunks)

            # If within limit, keep this hierarchy chunk unchanged
            else:

                chunks.append({
                    "document_id": document_id,
                    "chunk_id": str(uuid.uuid4()),
                    "parent_id": parent_id,
                    "level": level_name,
                    "start_page": level_chunk["start_page"],
                    "end_page": level_chunk["end_page"],
                    "text": text,
                    "token_count": token_count
                })

    return chunks

In [58]:
document_id = "BNS_2023"

chunks = recursive_chunking(
    pages=pages,
    hierarchy=structure_profile.hierarchy,
    document_id=document_id
)

print("Total chunks:", len(chunks))

Total chunks: 791


In [65]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")
print("Total chunks:", len(chunks))

token_counts = [len(encoding.encode(chunk["text"])) for chunk in chunks]

print("Smallest chunk:", min(token_counts))
print("Largest chunk:", max(token_counts))
print("Chunks over MAX_TOKENS:", sum(t > MAX_TOKENS for t in token_counts))

Total chunks: 791
Smallest chunk: 8
Largest chunk: 75991
Chunks over MAX_TOKENS: 44
